In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
!pip install langchain_community
!pip install langchain_openai
!pip install langchain-groq #freemodel
!pip install langgraph.checkpoint.sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
 

In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
import os
from google.colab import userdata

In [4]:
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
tool = TavilySearchResults(max_results=2)

/tmp/ipykernel_383/2968058647.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer) #using checkpointer for lang graph
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

### theory


1.   Persistence stores the graph state in a database (sqlite/postgres/redis) (called a checkpoint).
2.   Persistence stores agent state, not knowledge(like vector DB).
3.   Useful for the new user query which are following the previous ones.
4.   Agent State can contain

```
messages
tool outputs
variables
intermediate results
```





```
LangGraph
   ↓
loads saved state (messages, tool outputs)
   ↓
Agent workflow
   ↓
builds prompt using that state
   ↓
LLM generates next response
```


### How a thread is useful for loading appropriate agentstate from persistent storage
1.  Thread_id is like conversation identifier.

```
User message
      ↓
LangGraph loads checkpoint using thread_id
      ↓
State restored
      ↓
Graph node runs
      ↓
State updated
      ↓
Checkpointer saves new state
```





In [30]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# memory = SqliteSaver.from_conn_string(':memory:')
memory = SqliteSaver(sqlite3.connect(':memory:', check_same_thread=False))

In [31]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant")
zarvis_2_o = Agent(model, [tool], system=prompt, checkpointer=memory)

In [32]:
messages = [HumanMessage(content="What is the weather in haldwani?")]

In [33]:
thread = {"configurable": {"thread_id": "3"}}

### lets ask about weather query

In [34]:
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])
        print()

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'r0mpy8cds', 'function': {'arguments': '{"query":"haldwani weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 353, 'total_tokens': 375, 'completion_time': 0.039177426, 'completion_tokens_details': None, 'prompt_time': 0.021122663, 'prompt_tokens_details': None, 'queue_time': 0.005261488, 'total_time': 0.060300089}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce0f3-b766-7183-86f9-160f80211907-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'haldwani weather'}, 'id': 'r0mpy8cds', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 353, 'output_tokens': 22, 'total_tokens': 375})]

Calling: {'name': 'tavily_search_results_json',

### Lets ask about a followup query without weather context for same threadId:3

In [12]:
messages = [HumanMessage(content="What about in bageshwar?")]
thread = {"configurable": {"thread_id": "3"}}
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ca8bv6r5q', 'function': {'arguments': '{"query":"bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 3153, 'total_tokens': 3175, 'completion_time': 0.019391799, 'completion_tokens_details': None, 'prompt_time': 0.175768897, 'prompt_tokens_details': None, 'queue_time': 0.005849153, 'total_time': 0.195160696}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce0e6-9c70-7401-acbb-b23cb5c66226-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'bageshwar weather'}, 'id': 'ca8bv6r5q', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3153, 'output_tokens': 22, 'total_tokens': 3175})]}
Calling: {'name': 'tavily_se

### Lets ask about one more followup query without weather context for same threadId:3

In [13]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "3"}}
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Haldwani is warmer than Bageshwar.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 4790, 'total_tokens': 4803, 'completion_time': 0.014723638, 'completion_tokens_details': None, 'prompt_time': 0.275412475, 'prompt_tokens_details': None, 'queue_time': 0.006440951, 'total_time': 0.290136113}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce0e7-6560-72d3-987e-1737a4827d92-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4790, 'output_tokens': 13, 'total_tokens': 4803})]}


### Lets ask about a followup query without weather context for different threadId:4

In [14]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "4"}}
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jk09j978h', 'function': {'arguments': '{"query":"warmest planet in our solar system"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 348, 'total_tokens': 372, 'completion_time': 0.047880361, 'completion_tokens_details': None, 'prompt_time': 0.019157661, 'prompt_tokens_details': None, 'queue_time': 0.005309709, 'total_time': 0.067038022}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce0e8-1eb8-7b83-81db-65ce5f462e9e-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'warmest planet in our solar system'}, 'id': 'jk09j978h', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 348, 'output_tokens': 24, 'total_tokens': 372})]

### **Streaming Tokens**

stream() → streams graph outputs per node



astream_events() → streams low-level runtime events (including tokens)